In [1]:
# STANDIN A0
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-For-Cybersecurity-Lab-4.2-Robustness-attacks


train 267,984 rows   attack rate 0.1507
val    89,328 rows   attack rate 0.1507
test   89,329 rows   attack rate 0.1507


68 features: 58 continuous, 10 flag columns
Flag columns: Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count
  tree    loaded from models/tree.joblib
  logreg  loaded from models/logreg.joblib


  forest  loaded from models/forest.joblib


ATTACK_IDX: 50 detected attacks, forest probability 1.000-1.000
  attack types: {'DDoS': 25, 'DoS Hulk': 22, 'DoS GoldenEye': 2, 'SSH-Patator': 1}
EXPLAIN_IDX: 500 random test flows, 68 of them attacks
Setup complete: 20 shared names ready. The models are scored in step A2.


In [2]:
# STANDIN B1
# --- copy of step B1's add_noise cell (Track 1), so this notebook runs on its own ---
import matplotlib.pyplot as plt

# Columns that are never negative in the training data: clipping these at zero only removes
# impossible values that the noise created (a negative packet count). Columns that are already
# negative in real data (-1 = "not recorded") are left alone, so clipping never edits a real value.
NONNEG_COLS = [c for c in CONT_COLS if X_train[c].min() >= 0]


def add_noise(X, X_ref, level, seed=SEED, clip=True):
    """Noisy copy of X: Gaussian noise of width level * std(X_ref) on every continuous column.

    X_ref must be the training set (std measured there, never on the data being scored).
    The 0/1 flag columns are not touched. With clip=True, columns that are never negative in
    X_ref are clipped at zero after the noise is added.
    """
    noisy = X.copy()
    if level == 0:
        return noisy
    rng = np.random.default_rng(seed)
    std = X_ref[CONT_COLS].std().to_numpy()
    values = noisy[CONT_COLS].to_numpy(dtype=float)
    values += rng.standard_normal(values.shape) * (level * std)
    if clip:
        nonneg = np.isin(CONT_COLS, [c for c in CONT_COLS if X_ref[c].min() >= 0])
        values[:, nonneg] = np.maximum(values[:, nonneg], 0.0)
    noisy[CONT_COLS] = values
    return noisy


# Sanity checks: flags untouched, level 0 changes nothing, clipped columns never negative.
_check = add_noise(X_test.iloc[:1000], X_train, 0.5)
assert _check[FLAG_COLS].equals(X_test.iloc[:1000][FLAG_COLS])
assert add_noise(X_test.iloc[:1000], X_train, 0.0).equals(X_test.iloc[:1000])
assert (_check[NONNEG_COLS] >= 0).all().all()
print(f"{len(NONNEG_COLS)} of {len(CONT_COLS)} continuous columns are clipped at zero")

47 of 58 continuous columns are clipped at zero


<!-- STEP D2 -->
### Step D2: Is the explanation stable under noise?

**Stable** means: run the explainer on almost the same input and get almost the same answer. We
explain the same 200 test flows (the first 200 of `EXPLAIN_IDX`, a subset of D1's 500) twice with
TreeSHAP, once clean and once with B1's 5% noise. For each run we rank the features by mean |SHAP|
for the attack class, and compare the two rankings with the **Spearman rank correlation** (1 = same
order, 0 = unrelated). Above 0.90 counts as stable.

Stable is not the same as *faithful* (lab section 3.5): an explanation can be perfectly stable and
still describe the wrong thing. Part F tests faithfulness.

B1 showed that 5% noise already makes the forest call almost every attack benign, so we also run two
much smaller levels (0.1% and 1%) for context. They let us tell apart "SHAP is unstable" from "the
model itself decided differently".

In [3]:
# STEP D2
import shap
from scipy.stats import spearmanr

d2_flows = X_test.iloc[EXPLAIN_IDX[:200]]
d2_y = y_test.iloc[EXPLAIN_IDX[:200]]
d2_explainer = shap.TreeExplainer(rf)


def attack_shap(X):
    """SHAP values for the attack class: one row per flow, one column per feature."""
    return d2_explainer(X).values[:, :, 1]


def ranking(values, rows=None):
    """Mean |SHAP| per feature over the given rows (all rows by default)."""
    v = values if rows is None else values[rows]
    return pd.Series(np.abs(v).mean(axis=0), index=FEATURES)


start = time.time()
d2_levels = [0.001, 0.01, 0.05]
shap_clean = attack_shap(d2_flows)
shap_noisy = {level: attack_shap(add_noise(d2_flows, X_train, level)) for level in d2_levels}
rank_clean = ranking(shap_clean)
d2_rankings = {level: ranking(v) for level, v in shap_noisy.items()}
print(f"{len(d2_flows)} flows ({int(d2_y.sum())} attacks), SHAP 4 times in {time.time() - start:.0f} s")

200 flows (24 attacks), SHAP 4 times in 347 s


In [4]:
# STEP D2
rows = []
pred_clean = rf.predict(d2_flows)
for level, rank_noisy in d2_rankings.items():
    rho, _ = spearmanr(rank_clean, rank_noisy)
    top10_overlap = len(set(rank_clean.nlargest(10).index) & set(rank_noisy.nlargest(10).index))
    pred_noisy = rf.predict(add_noise(d2_flows, X_train, level))
    rows.append({"noise level": level, "spearman": rho, "top-10 overlap": top10_overlap,
                 "top-5 identical": list(rank_clean.nlargest(5).index) == list(rank_noisy.nlargest(5).index),
                 "flows whose prediction changed": int((pred_clean != pred_noisy).sum()),
                 "attacks still detected": f"{int(pred_noisy[d2_y.to_numpy() == 1].sum())} of {int(d2_y.sum())}"})
d2_summary = pd.DataFrame(rows).set_index("noise level")
d2_summary.to_csv("results/tables/D2_stability_by_level.csv", float_format="%.4f")
d2_summary.round(4)

,spearman,top-10 overlap,top-5 identical,flows whose prediction changed,attacks still detected
noise level,,,,,
0.001,0.9071,9,False,5,19 of 24
0.010,0.9048,7,False,19,5 of 24
0.050,0.8991,7,False,23,1 of 24


In [5]:
# STEP D2
# The lab's comparison: clean vs 5% noise, both top-5 lists and the full rankings.
rank_noisy5 = d2_rankings[0.05]
rho5, _ = spearmanr(rank_clean, rank_noisy5)
print(f"Spearman rank correlation, clean vs 5% noise: {rho5:.3f}  "
      f"({'stable' if rho5 > 0.90 else 'NOT stable'} by the lab's 0.90 rule)\n")
top5 = pd.DataFrame({"clean": rank_clean.nlargest(5).index,
                     "5% noise": rank_noisy5.nlargest(5).index}, index=range(1, 6))
print(top5.to_string())

d2_table = pd.DataFrame({
    "mean_abs_shap_clean": rank_clean, "mean_abs_shap_noisy5": rank_noisy5,
    "rank_clean": rank_clean.rank(ascending=False).astype(int),
    "rank_noisy5": rank_noisy5.rank(ascending=False).astype(int),
}).sort_values("rank_clean")
d2_table.index.name = "feature"
d2_table.to_csv("results/tables/D2_shap_stability.csv", float_format="%.5f")
print(f"\nTotal mean |SHAP|: clean {rank_clean.sum():.3f}, 5% noise {rank_noisy5.sum():.3f}")

Spearman rank correlation, clean vs 5% noise: 0.899  (NOT stable by the lab's 0.90 rule)

                    clean               5% noise
1   Bwd Packet Length Std  Bwd Packet Length Std
2    Avg Bwd Segment Size          Bwd Packets/s
3  Bwd Packet Length Mean   Avg Bwd Segment Size
4     Average Packet Size    Average Packet Size
5  Packet Length Variance      Bwd Header Length

Total mean |SHAP|: clean 0.274, 5% noise 0.328


In [6]:
# STEP D2
# Two reasons the overall correlation can hide change: most of the 200 flows are benign, and most of
# the 68 features have near-zero SHAP, so their order reshuffles at any noise level.
is_attack = d2_y.to_numpy() == 1
top20 = rank_clean.nlargest(20).index
rows = []
for level in d2_levels:
    noisy = shap_noisy[level]
    rows.append({
        "noise level": level,
        "all 200 flows, 68 features": spearmanr(rank_clean, d2_rankings[level])[0],
        "all 200 flows, clean top-20 features": spearmanr(rank_clean[top20], d2_rankings[level][top20])[0],
        "24 attacks only, 68 features": spearmanr(ranking(shap_clean, is_attack), ranking(noisy, is_attack))[0],
        "176 benign only, 68 features": spearmanr(ranking(shap_clean, ~is_attack), ranking(noisy, ~is_attack))[0],
        "attacks: mean SHAP sum (clean -> noisy)":
            f"{shap_clean[is_attack].sum(axis=1).mean():+.3f} -> {noisy[is_attack].sum(axis=1).mean():+.3f}",
    })
d2_split = pd.DataFrame(rows).set_index("noise level")
d2_split.to_csv("results/tables/D2_stability_split.csv", float_format="%.4f")
print("Attack-flow top 5, clean vs 5% noise:")
print(pd.DataFrame({"clean": ranking(shap_clean, is_attack).nlargest(5).index,
                    "5% noise": ranking(shap_noisy[0.05], is_attack).nlargest(5).index},
                   index=range(1, 6)).to_string())
d2_split.round(3)

Attack-flow top 5, clean vs 5% noise:
                    clean                     5% noise
1   Bwd Packet Length Std        Bwd Packet Length Std
2    Avg Bwd Segment Size            Subflow Bwd Bytes
3  Packet Length Variance  Total Length of Bwd Packets
4  Bwd Packet Length Mean       Packet Length Variance
5       Packet Length Std         Avg Bwd Segment Size


,"all 200 flows, 68 features","all 200 flows, clean top-20 features","24 attacks only, 68 features","176 benign only, 68 features",attacks: mean SHAP sum (clean -> noisy)
noise level,,,,,
0.001,0.907,0.908,0.958,0.863,+0.842 -> +0.492
0.010,0.905,0.669,0.948,0.835,+0.842 -> +0.274
0.050,0.899,0.496,0.949,0.830,+0.842 -> +0.216


<!-- STEP D2 -->
**What we found.** By the lab's test the SHAP ranking is right at the border: the Spearman correlation
between the clean and the 5%-noise ranking (68 features, 200 flows) is **0.899**, just under 0.90. The
top-ranked feature stays the same (`Bwd Packet Length Std`), and 3 of the top 5 survive.

| Noise | Spearman, 68 features | Spearman, top-20 features | Spearman, attacks only | Attacks still detected |
|---|---|---|---|---|
| 0.1% | 0.907 | 0.908 | 0.958 | 19 of 24 |
| 1% | 0.905 | 0.669 | 0.948 | 5 of 24 |
| 5% | 0.899 | **0.496** | 0.949 | **1 of 24** |

That single number hides two things.

1. **The important part of the ranking is not stable.** Over the 20 features that carry most of the
   SHAP weight, the correlation falls to 0.67 at 1% noise and 0.50 at 5%. The 68-feature number stays
   near 0.90 only because most features have almost no SHAP weight and stay at the bottom of both
   rankings, so their agreement props up the overall correlation. A ranking can look "stable" mainly
   because the unimportant features stay unimportant.
2. **A stable ranking can hide a flipped decision.** For the 24 attacks the ranking hardly changes
   (0.95 at every level): the explanation keeps naming the same features (`Bwd Packet Length Std`,
   `Avg Bwd Segment Size`, `Packet Length Variance`). But the total push those features give towards
   "attack" falls from +0.84 to +0.22, and the forest now calls 23 of the 24 attacks benign. An
   analyst who only looked at the ranking would see the same explanation for a decision that has
   reversed.

So the SHAP explanation is stable in the narrow sense the lab measures (borderline at 5%), but that
stability says nothing about whether the explanation, or the model, is right. This is lab section
3.5's warning in practice: stable is not the same as faithful. Part F tests faithfulness directly.
The 200 flows are a subset of D1's 500 (24 attacks); their clean ranking has the same leader as
D1's.